# Linear + MLP Position Probes
Trains both a **linear** and a **shallow MLP** probe on the frozen GRU hidden state
and compares their ability to decode 2D object positions.

Two matching modes (`USE_HUNGARIAN`):
- `True`  — Hungarian matching; use with **random reflectivities** datasets.
- `False` — Direct MSE on fixed object order; use with **fixed_reflectivities** datasets.

The linear probe uses an exact least-squares solve when `USE_HUNGARIAN=False`;
both probes use gradient descent when `USE_HUNGARIAN=True`.

In [ ]:
import sys

sys.path.insert(0, ".")

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MPoly
from tqdm.notebook import tqdm

import helpers.nb_utils as nb_utils
import helpers.nb_viz as nb_viz

## Config — edit these

In [ ]:
CHECKPOINT_PATH = "../runs/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
TRAIN_H5_PATH = "../datasets/3_fixed_refl_inview_brighter_train/dataset.h5"
TEST_H5_PATH = "../datasets/3_fixed_refl_inview_brighter_eval/dataset.h5"

NUM_EXTRACTOR_TRAIN = 10_000
DEVICE = "cuda"
BATCH_SIZE = 256
NUM_WORKERS = 12
LR = 1e-3
N_EPOCHS = 30
MLP_HIDDEN = 128

# True  → Hungarian matching (random reflectivities dataset)
# False → Direct MSE on fixed object order (fixed_reflectivities dataset)
USE_HUNGARIAN = False

# True  → autoregressive rollout (model feeds its own predictions after NUM_OBS steps)
# False → teacher forcing (ground-truth observations at every step)
USE_AUTOREGRESSIVE = False
NUM_OBS = 10  # context steps before AR rollout begins (only used when USE_AUTOREGRESSIVE=True)

# Probe colors used in comparison plots
LINEAR_COLOR = "#0072B2"  # Okabe-Ito blue
MLP_COLOR = "#D55E00"  # Okabe-Ito vermilion

## Load frozen GRU

In [ ]:
model, ckpt_info = nb_utils.load_model(CHECKPOINT_PATH, DEVICE)
hidden_size = ckpt_info["model_config"]["hidden_size"]

print(f"GRU epoch    : {ckpt_info['epoch']}")
print(f"GRU val loss : {ckpt_info['val_loss']:.6f}")
print(f"Hidden size  : {hidden_size}")

with h5py.File(TRAIN_H5_PATH, "r") as f:
    n_total = f["obs_intensity"].shape[0]
    max_obj = f["positions"].shape[2]
    T_frames = f["obs_intensity"].shape[1]

print(f"\nDataset      : {n_total:,} samples, T={T_frames}, max_obj={max_obj}")

rng = np.random.default_rng(42)
train_idx = rng.choice(n_total, size=min(NUM_EXTRACTOR_TRAIN, n_total), replace=False)
print(f"Probe train  : {len(train_idx):,} samples")

In [ ]:
# ── Hidden-state extraction helper ────────────────────────────────────────────
# Wraps teacher-forcing and autoregressive rollout behind a single call.
# All callers below use _get_h(model, obs, DEVICE) — swap USE_AUTOREGRESSIVE
# in the config cell to change which mode every cell uses.


@torch.no_grad()
def _get_hidden_states_ar(model, obs, device, num_obs):
    """Autoregressive rollout.

    Teacher-forces the first *num_obs* steps, then feeds the model's own
    predicted observation back as input for the remaining steps.

    Parameters
    ----------
    obs : (B, T, R) float tensor
    num_obs : int  — steps of ground-truth context before AR begins

    Returns
    -------
    h : (B, T-1, H)  hidden states at each step (same shape as teacher-forcing)
    """
    obs = obs.to(device)
    B, T, R = obs.shape
    h_list, gru_h, pred = [], None, None
    for t in range(T - 1):
        inp = obs[:, t, :] if t < num_obs else pred
        x = F.relu(model.encoder(inp)).unsqueeze(1)  # (B, 1, enc_dim)
        h_out, gru_h = model.gru(x, gru_h)  # (B, 1, H)
        h_list.append(h_out.squeeze(1))  # (B, H)
        pred = model.decoder(h_out.squeeze(1))  # (B, R)  next-step prediction
    return torch.stack(h_list, dim=1)  # (B, T-1, H)


def _get_h(model, obs, device):
    """Return hidden states using the mode set by USE_AUTOREGRESSIVE."""
    if USE_AUTOREGRESSIVE:
        return _get_hidden_states_ar(model, obs, device, NUM_OBS)
    return nb_utils.get_hidden_states(model, obs, device)


mode_str_ar = f"AR(ctx={NUM_OBS})" if USE_AUTOREGRESSIVE else "TF"
print(f"Hidden-state mode: {mode_str_ar}")

In [ ]:
# ── Dataset sanity check — waterfall grid ────────────────────────────────────
from pim.viz import make_waterfall

N_SANITY = 6
fig, axes = plt.subplots(
    1, N_SANITY, figsize=(N_SANITY * 2.2, 3.5), facecolor=nb_viz._BG_HEX
)
fig.suptitle(
    f"TRAIN DATASET:  {TRAIN_H5_PATH}",
    fontsize=13,
    fontweight="bold",
    color=nb_viz._TEXT_COLOR,
    y=1.02,
)

for ax, idx in zip(axes, range(N_SANITY)):
    scene, obs_depth, obs_id, obs_intensity = nb_utils.load_sample(TRAIN_H5_PATH, idx)
    wf = make_waterfall(obs_depth, obs_id, obs_intensity, scene)
    ax.imshow(wf, aspect="auto", origin="upper", interpolation="nearest")
    refl_str = "  ".join(f"{r:.2f}" for r in scene.reflectivities)
    ax.set_title(f"#{idx}  [{refl_str}]", color=nb_viz._TEXT_COLOR, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ── GRU training curve ────────────────────────────────────────────────────────
import json
from pathlib import Path

metrics_path = Path(CHECKPOINT_PATH).parent / "metrics.jsonl"
metrics = [
    json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()
]

epochs = [m["epoch"] for m in metrics]
train_loss = [m["train_loss"] for m in metrics]
val_loss = [m["val_loss"] for m in metrics]
best_epoch = ckpt_info["epoch"]
run_name = Path(CHECKPOINT_PATH).parent.name

fig, ax = plt.subplots(figsize=(9, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)

ax.plot(epochs, train_loss, color=LINEAR_COLOR, linewidth=1.6, label="train")
ax.plot(epochs, val_loss, color=MLP_COLOR, linewidth=1.6, label="val")
ax.axvline(
    best_epoch,
    color=nb_viz._TICK_COLOR,
    linewidth=1.0,
    linestyle="--",
    alpha=0.6,
    label=f"best (epoch {best_epoch})",
)

ax.set_yscale("log")
ax.set_xlabel("epoch", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel("loss (log scale)", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title(
    f"best val loss: {ckpt_info['val_loss']:.6f}  @  epoch {best_epoch}",
    color=nb_viz._TEXT_COLOR,
    fontsize=11,
)
ax.legend(
    labelcolor=nb_viz._TEXT_COLOR,
    facecolor=nb_viz._BG_HEX,
    edgecolor=nb_viz._TICK_COLOR,
    fontsize=10,
)

fig.suptitle(
    f"MODEL:  {run_name}",
    fontsize=15,
    fontweight="bold",
    color=nb_viz._TEXT_COLOR,
    y=1.02,
)

plt.tight_layout()
plt.show()

## Linear probe

In [ ]:
class LinearExtractor(nn.Module):
    """hidden_size → (max_obj, 2) via a single linear layer."""

    def __init__(self, hidden_size: int, max_obj: int) -> None:
        super().__init__()
        self.linear = nn.Linear(hidden_size, max_obj * 2)
        self.max_obj = max_obj

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.linear(h).reshape(*h.shape[:-1], self.max_obj, 2)


linear_extractor = LinearExtractor(hidden_size, max_obj).to(DEVICE)
linear_optimizer = torch.optim.Adam(linear_extractor.parameters(), lr=LR)
print(
    f"Linear extractor params: {sum(p.numel() for p in linear_extractor.parameters()):,}"
)

### Train linear probe

In [ ]:
def hungarian_mse(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    loss_01 = ((pred - target) ** 2).mean(dim=(1, 2))
    loss_10 = ((pred - target[:, [1, 0], :]) ** 2).mean(dim=(1, 2))
    return torch.minimum(loss_01, loss_10).mean()


train_loader = nb_utils.build_loader(
    TRAIN_H5_PATH,
    train_idx,
    keys=("obs_intensity", "positions", "is_visible"),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True,
)

if USE_HUNGARIAN:
    # ── Gradient descent ────────────────────────────────────────────────────
    linear_losses = []
    for epoch in tqdm(range(1, N_EPOCHS + 1), desc="linear epochs"):
        linear_extractor.train()
        running, n_batches = 0.0, 0
        for batch in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
            obs = batch["obs_intensity"].to(DEVICE)
            pos = batch["positions"].to(DEVICE)
            vis = batch["is_visible"].bool().to(DEVICE)
            B, T = obs.shape[0], obs.shape[1]
            h = _get_h(model, obs, DEVICE)
            vis_t = vis[:, :-1, :]
            pos_t = pos[:, :-1, :, :]
            all_vis = vis_t.all(dim=2)
            h_flat = h.reshape(B * (T - 1), hidden_size)
            pos_flat = pos_t.reshape(B * (T - 1), max_obj, 2)
            mask = all_vis.reshape(B * (T - 1))
            if mask.sum() == 0:
                continue
            pred = linear_extractor(h_flat[mask])
            loss = hungarian_mse(pred, pos_flat[mask])
            linear_optimizer.zero_grad()
            loss.backward()
            linear_optimizer.step()
            running += loss.item()
            n_batches += 1
        linear_losses.append(running / n_batches if n_batches else float("nan"))
    print(f"\nLinear final train loss: {linear_losses[-1]:.6f}")

else:
    # ── Exact least-squares ──────────────────────────────────────────────────
    all_h, all_pos = [], []
    linear_extractor.eval()
    with torch.no_grad():
        for batch in tqdm(train_loader, desc="collecting hidden states"):
            obs = batch["obs_intensity"].to(DEVICE)
            pos = batch["positions"].to(DEVICE)
            vis = batch["is_visible"].bool().to(DEVICE)
            B, T = obs.shape[0], obs.shape[1]
            h = _get_h(model, obs, DEVICE)
            vis_t = vis[:, :-1, :]
            pos_t = pos[:, :-1, :, :]
            all_vis = vis_t.all(dim=2)
            h_flat = h.reshape(B * (T - 1), hidden_size).cpu().numpy()
            pos_flat = pos_t.reshape(B * (T - 1), max_obj * 2).cpu().numpy()
            mask = all_vis.reshape(B * (T - 1)).cpu().numpy()
            all_h.append(h_flat[mask])
            all_pos.append(pos_flat[mask])
    X = np.concatenate(all_h, axis=0)
    Y = np.concatenate(all_pos, axis=0)
    X_aug = np.concatenate([X, np.ones((len(X), 1), dtype=X.dtype)], axis=1)
    W, _, rank, _ = np.linalg.lstsq(X_aug, Y, rcond=None)
    with torch.no_grad():
        linear_extractor.linear.weight.copy_(torch.from_numpy(W[:-1].T).float())
        linear_extractor.linear.bias.copy_(torch.from_numpy(W[-1]).float())
    train_mse = np.mean((X_aug @ W - Y) ** 2)
    print(
        f"Exact regression: {len(X):,} frames  |  rank {rank}  |  train MSE {train_mse:.6f}"
    )

In [ ]:
if USE_HUNGARIAN:
    loss_label = "MSE (matched)"
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=nb_viz._BG_HEX)
    nb_viz.style_ax(ax)
    ax.plot(range(1, N_EPOCHS + 1), linear_losses, color=LINEAR_COLOR, linewidth=1.8)
    ax.set_xlabel("epoch", color=nb_viz._TEXT_COLOR, fontsize=11)
    ax.set_ylabel(loss_label, color=nb_viz._TEXT_COLOR, fontsize=11)
    ax.set_title("Linear probe — training loss", color=nb_viz._TEXT_COLOR, fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    loss_label = "MSE (direct)"
    print("Exact regression — no epoch curve.")

## MLP probe

In [ ]:
class MLPExtractor(nn.Module):
    """hidden_size → MLP_HIDDEN (ReLU) → (max_obj, 2)."""

    def __init__(self, hidden_size: int, mlp_hidden: int, max_obj: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, max_obj * 2),
        )
        self.max_obj = max_obj

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.net(h).reshape(*h.shape[:-1], self.max_obj, 2)


mlp_extractor = MLPExtractor(hidden_size, MLP_HIDDEN, max_obj).to(DEVICE)
mlp_optimizer = torch.optim.Adam(mlp_extractor.parameters(), lr=LR)
print(f"MLP extractor params : {sum(p.numel() for p in mlp_extractor.parameters()):,}")
print(f"Architecture         : {hidden_size} → {MLP_HIDDEN} → {max_obj * 2}")

### Train MLP probe

In [ ]:
mlp_losses = []
for epoch in tqdm(range(1, N_EPOCHS + 1), desc="MLP epochs"):
    mlp_extractor.train()
    running, n_batches = 0.0, 0
    for batch in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
        obs = batch["obs_intensity"].to(DEVICE)
        pos = batch["positions"].to(DEVICE)
        vis = batch["is_visible"].bool().to(DEVICE)
        B, T = obs.shape[0], obs.shape[1]
        h = _get_h(model, obs, DEVICE)
        vis_t = vis[:, :-1, :]
        pos_t = pos[:, :-1, :, :]
        all_vis = vis_t.all(dim=2)
        h_flat = h.reshape(B * (T - 1), hidden_size)
        pos_flat = pos_t.reshape(B * (T - 1), max_obj, 2)
        mask = all_vis.reshape(B * (T - 1))
        if mask.sum() == 0:
            continue
        pred = mlp_extractor(h_flat[mask])
        if USE_HUNGARIAN:
            loss = hungarian_mse(pred, pos_flat[mask])
        else:
            loss = F.mse_loss(pred, pos_flat[mask])
        mlp_optimizer.zero_grad()
        loss.backward()
        mlp_optimizer.step()
        running += loss.item()
        n_batches += 1
    mlp_losses.append(running / n_batches if n_batches else float("nan"))

print(f"\nMLP final train loss: {mlp_losses[-1]:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.plot(range(1, N_EPOCHS + 1), mlp_losses, color=MLP_COLOR, linewidth=1.8)
ax.set_xlabel("epoch", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel(loss_label, color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title(
    f"MLP probe — training loss  (hidden={MLP_HIDDEN})",
    color=nb_viz._TEXT_COLOR,
    fontsize=12,
)
ax.set_xscale("log")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## Evaluation on test set

In [ ]:
test_loader = nb_utils.build_loader(
    TEST_H5_PATH,
    indices=None,
    keys=("obs_intensity", "positions", "is_visible"),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

linear_extractor.eval()
mlp_extractor.eval()

linear_per_t_sum = np.zeros(T_frames - 1)
linear_per_t_count = np.zeros(T_frames - 1)
linear_per_obj_sum = np.zeros(max_obj)
mlp_per_t_sum = np.zeros(T_frames - 1)
mlp_per_obj_sum = np.zeros(max_obj)
per_obj_count = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="test eval"):
        obs = batch["obs_intensity"].to(DEVICE)
        pos = batch["positions"].to(DEVICE)
        vis = batch["is_visible"].bool().to(DEVICE)
        B, T = obs.shape[0], obs.shape[1]
        h = _get_h(model, obs, DEVICE)  # one pass, shared by both probes
        vis_t = vis[:, :-1, :]
        pos_t = pos[:, :-1, :, :]
        all_vis = vis_t.all(dim=2)
        h_flat = h.reshape(B * (T - 1), hidden_size)
        pos_flat = pos_t.reshape(B * (T - 1), max_obj, 2)
        mask = all_vis.reshape(B * (T - 1))
        t_idx = (
            torch.arange(T - 1, device=DEVICE).unsqueeze(0).expand(B, -1).reshape(-1)
        )
        if mask.sum() == 0:
            continue
        gt_pos = pos_flat[mask]
        t_vis = t_idx[mask].cpu().numpy()

        for extractor, per_t_sum, per_obj_sum in [
            (linear_extractor, linear_per_t_sum, linear_per_obj_sum),
            (mlp_extractor, mlp_per_t_sum, mlp_per_obj_sum),
        ]:
            pred = extractor(h_flat[mask])
            if USE_HUNGARIAN:
                loss_01 = ((pred - gt_pos) ** 2).mean(dim=(1, 2))
                loss_10 = ((pred - gt_pos[:, [1, 0], :]) ** 2).mean(dim=(1, 2))
                per_sample_mse = torch.minimum(loss_01, loss_10).cpu().numpy()
                swap = loss_10 < loss_01
                gt_matched = torch.where(
                    swap[:, None, None], gt_pos[:, [1, 0], :], gt_pos
                )
            else:
                per_sample_mse = ((pred - gt_pos) ** 2).mean(dim=(1, 2)).cpu().numpy()
                gt_matched = gt_pos
            per_obj_batch = ((pred - gt_matched) ** 2).mean(dim=2).cpu().numpy()
            per_obj_sum += per_obj_batch.sum(axis=0)
            np.add.at(per_t_sum, t_vis, per_sample_mse)

        per_obj_count += mask.sum().item()
        np.add.at(linear_per_t_count, t_vis, 1)

mlp_per_t_count = linear_per_t_count.copy()  # masks are identical
linear_per_t_mse = np.where(
    linear_per_t_count > 0, linear_per_t_sum / linear_per_t_count, np.nan
)
mlp_per_t_mse = np.where(mlp_per_t_count > 0, mlp_per_t_sum / mlp_per_t_count, np.nan)
linear_per_obj_mse = linear_per_obj_sum / max(per_obj_count, 1)
mlp_per_obj_mse = mlp_per_obj_sum / max(per_obj_count, 1)

mode_str = "matched" if USE_HUNGARIAN else "direct"
print(f"Hidden-state mode: {mode_str_ar}")
print(f"Linear — overall MSE ({mode_str}): {np.nanmean(linear_per_t_mse):.6f}")
for i in range(max_obj):
    print(f"  Object {i}: {linear_per_obj_mse[i]:.6f}")
print(f"MLP    — overall MSE ({mode_str}): {np.nanmean(mlp_per_t_mse):.6f}")
for i in range(max_obj):
    print(f"  Object {i}: {mlp_per_obj_mse[i]:.6f}")

In [ ]:
# ── MSE vs context length ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.plot(
    range(T_frames - 1),
    linear_per_t_mse,
    color=LINEAR_COLOR,
    linewidth=1.8,
    label="linear",
)
ax.plot(range(T_frames - 1), mlp_per_t_mse, color=MLP_COLOR, linewidth=1.8, label="MLP")
ax.set_xlabel("context length (timestep)", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel(f"mean {loss_label} (visible)", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title("Position MSE vs context length", color=nb_viz._TEXT_COLOR, fontsize=12)
ax.legend(
    labelcolor=nb_viz._TEXT_COLOR,
    facecolor=nb_viz._BG_HEX,
    edgecolor=nb_viz._TICK_COLOR,
    fontsize=10,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-object MSE bar chart ──────────────────────────────────────────────────
x = np.arange(max_obj)
width = 0.35

fig, ax = plt.subplots(figsize=(max(5, max_obj * 2), 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.bar(
    x - width / 2,
    linear_per_obj_mse,
    width,
    label="linear",
    color=LINEAR_COLOR,
    alpha=0.85,
)
ax.bar(x + width / 2, mlp_per_obj_mse, width, label="MLP", color=MLP_COLOR, alpha=0.85)
ax.set_xlabel("object", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel(f"mean {loss_label}", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title("Per-object position MSE", color=nb_viz._TEXT_COLOR, fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([f"obj {i}" for i in range(max_obj)])
ax.legend(
    labelcolor=nb_viz._TEXT_COLOR,
    facecolor=nb_viz._BG_HEX,
    edgecolor=nb_viz._TICK_COLOR,
    fontsize=10,
)
plt.tight_layout()
plt.show()

## Single-sample visualization

In [ ]:
from pim.sim import Scene

# Visualization
VIZ_SAMPLE_IDX = 1

scene, obs_depth, obs_id, obs_intensity = nb_utils.load_sample(
    TEST_H5_PATH, VIZ_SAMPLE_IDX
)

with h5py.File(TEST_H5_PATH, "r") as f:
    vis_sample = f["is_visible"][VIZ_SAMPLE_IDX].astype(bool)
    pos_sample = f["positions"][VIZ_SAMPLE_IDX].astype(np.float32)

obs_t = torch.from_numpy(obs_intensity).float().unsqueeze(0).to(DEVICE)

linear_extractor.eval()
mlp_extractor.eval()
with torch.no_grad():
    h_sample = _get_h(model, obs_t, DEVICE)  # (1, T-1, H)
    linear_pred_pos = (
        linear_extractor(h_sample.squeeze(0)).cpu().numpy()
    )  # (T-1, max_obj, 2)
    mlp_pred_pos = mlp_extractor(h_sample.squeeze(0)).cpu().numpy()

gt_pos = pos_sample[:-1]  # (T-1, max_obj, 2)
vis_tm1 = vis_sample[:-1]  # (T-1, max_obj)
all_vis = vis_tm1.all(axis=1)
timesteps = np.arange(T_frames - 1)


# per-object MSE for each probe
def _sample_per_obj_mse(pred, gt, vis_mask, use_hungarian):
    if use_hungarian and vis_mask.any():
        err_01 = ((pred[vis_mask] - gt[vis_mask]) ** 2).mean()
        err_10 = ((pred[vis_mask] - gt[vis_mask][:, [1, 0], :]) ** 2).mean()
        gt_m = gt if err_01 <= err_10 else gt[:, [1, 0], :]
    else:
        gt_m = gt
    return np.array(
        [
            (
                ((pred[vis_mask, i] - gt_m[vis_mask, i]) ** 2).mean()
                if vis_mask.any()
                else np.nan
            )
            for i in range(pred.shape[1])
        ]
    )


linear_sample_mse = _sample_per_obj_mse(linear_pred_pos, gt_pos, all_vis, USE_HUNGARIAN)
mlp_sample_mse = _sample_per_obj_mse(mlp_pred_pos, gt_pos, all_vis, USE_HUNGARIAN)

print(
    f"Sample {VIZ_SAMPLE_IDX} — hidden-state mode: {mode_str_ar}  |  matching: {mode_str}"
)
for i in range(max_obj):
    print(
        f"  Object {i}:  linear {linear_sample_mse[i]:.4f}   MLP {mlp_sample_mse[i]:.4f}"
    )

In [ ]:
# ── Trajectory plot: GT + both probe predictions ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor=nb_viz._BG_HEX)
mse_str = "  |  ".join(
    f"obj {i}: lin {linear_sample_mse[i]:.3f} / mlp {mlp_sample_mse[i]:.3f}"
    for i in range(max_obj)
)
fig.suptitle(
    f"Sample {VIZ_SAMPLE_IDX}  [{mse_str}]\n" "solid=actual   ×=linear   o=MLP",
    color=nb_viz._TEXT_COLOR,
    fontsize=11,
)

for ax, coord, label in zip(axes, [0, 1], ["x", "y (depth)"]):
    nb_viz.style_ax(ax)
    for obj in range(min(max_obj, scene.positions.shape[1])):
        color = nb_viz.plot_color(scene.colors[obj])
        vis_obj = vis_tm1[:, obj]
        ax.plot(
            timesteps,
            gt_pos[:, obj, coord],
            color=color,
            linewidth=1.8,
            label=f"obj {obj} actual",
        )
        ax.scatter(
            timesteps[vis_obj],
            linear_pred_pos[vis_obj, obj, coord],
            color=color,
            s=18,
            marker="x",
            alpha=0.75,
            label=f"obj {obj} linear",
        )
        ax.scatter(
            timesteps[vis_obj],
            mlp_pred_pos[vis_obj, obj, coord],
            color=color,
            s=18,
            marker="o",
            alpha=0.75,
            label=f"obj {obj} MLP",
        )
    ax.set_xlabel("timestep", color=nb_viz._TEXT_COLOR, fontsize=10)
    ax.set_ylabel(label, color=nb_viz._TEXT_COLOR, fontsize=10)
    ax.set_title(label, color=nb_viz._TEXT_COLOR, fontsize=11)
    ax.legend(
        labelcolor=nb_viz._TEXT_COLOR,
        facecolor=nb_viz._BG_HEX,
        edgecolor=nb_viz._TICK_COLOR,
        fontsize=7,
        ncol=2,
    )

plt.tight_layout()
plt.show()

In [ ]:
# ── 2D scene overlay ──────────────────────────────────────────────────────────
cfg = scene.config
_dbg26 = nb_viz._DARK_BG_HEX
_dtc26 = nb_viz._DARK_TEXT_COLOR
_dfe26 = nb_viz._DARK_FRUSTUM_EDGE

fig, ax = plt.subplots(figsize=(6, 7), facecolor=_dbg26)
nb_viz.style_ax(ax, dark=True)
ax.set_xlim(-cfg.x_far - 0.7, cfg.x_far + 0.7)
ax.set_ylim(cfg.y_near - 0.7, cfg.y_far + 0.7)
ax.set_aspect("equal")
ax.set_xlabel("x", color=_dtc26, fontsize=10)
ax.set_ylabel("depth  y", color=_dtc26, fontsize=10)
ax.set_title(
    f"Sample {VIZ_SAMPLE_IDX} — 2D trajectories\nsolid=actual   ×=linear   o=MLP",
    color=_dtc26,
    fontsize=11,
)

corners = np.array([
    [-cfg.x_near, cfg.y_near], [cfg.x_near, cfg.y_near],
    [cfg.x_far,   cfg.y_far],  [-cfg.x_far, cfg.y_far],
])
ax.add_patch(MPoly(corners, closed=True, fill=False,
                   edgecolor=_dfe26, linewidth=1.5, zorder=1))

for obj in range(min(max_obj, scene.positions.shape[1])):
    color = scene.colors[obj]   # original sim colors — great on dark backgrounds
    vis_obj = vis_tm1[:, obj]
    ax.plot(gt_pos[:, obj, 0], gt_pos[:, obj, 1],
            color=color, linewidth=1.5, alpha=0.6, zorder=2)
    ax.scatter(linear_pred_pos[vis_obj, obj, 0], linear_pred_pos[vis_obj, obj, 1],
               color=color, s=18, marker="x", zorder=3, alpha=0.9,
               label=f"obj {obj} linear")
    ax.scatter(mlp_pred_pos[vis_obj, obj, 0], mlp_pred_pos[vis_obj, obj, 1],
               color=color, s=18, marker="o", zorder=3, alpha=0.9,
               label=f"obj {obj} MLP")

ax.legend(labelcolor=_dtc26, facecolor=_dbg26, edgecolor=nb_viz._DARK_TICK_COLOR,
          fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

---
## Rollout coherence
Warm the GRU up on `ROLLOUT_N_CONTEXT` real observations, then run it autoregressively for
`ROLLOUT_N_ROLLOUT` steps. The coherence metric and visualization are evaluated only on the
rollout phase — the question is whether the hidden state stays geometrically meaningful once
it is no longer anchored by real observations.

In [ ]:
ROLLOUT_N_CONTEXT = 10   # real observations for GRU warm-up
ROLLOUT_N_ROLLOUT = 20   # autoregressive steps after warm-up
ROLLOUT_N_EVAL    = 500  # samples for population-level coherence score
ROLLOUT_VIZ_IDX   = 0    # sample index for visualization

In [ ]:
@torch.no_grad()
def _collect_rollout(model, obs_np, n_context, n_rollout, device):
    """Warm up on n_context real observations (teacher forcing), then AR roll n_rollout steps.

    Returns
    -------
    h_ctx    : (n_context, H)  hidden states during context
    h_roll   : (n_rollout, H)  hidden states during rollout
    obs_roll : (n_rollout, R)  model-predicted observations during rollout

    Uses h_next[-1] (last GRU layer) as the probe-compatible hidden state —
    for a single-step GRU input this equals the GRU output, matching what the probes
    were trained on.
    """
    obs = torch.from_numpy(obs_np).float().unsqueeze(0).to(device)  # (1, T, R)
    h = None
    h_ctx_list = []

    for t in range(n_context):
        pred, h = model.step(obs[:, t, :], h)
        h_ctx_list.append(h[-1, 0])  # (H,)

    h_roll_list, obs_roll_list = [], []
    x = pred
    for _ in range(n_rollout):
        pred, h = model.step(x, h)
        h_roll_list.append(h[-1, 0])   # (H,)
        obs_roll_list.append(pred[0].cpu())  # (R,)
        x = pred

    return (
        torch.stack(h_ctx_list).cpu().numpy(),   # (n_context, H)
        torch.stack(h_roll_list).cpu().numpy(),  # (n_rollout, H)
        torch.stack(obs_roll_list).numpy(),      # (n_rollout, R)
    )


def rollout_coherence(pos_rollout):
    """Dimensionless trajectory coherence score.

    Parameters
    ----------
    pos_rollout : (N, n_obj, 2)

    Returns
    -------
    score      : float — mean relative acceleration (lower = more linear/coherent; 0 = perfect)
    per_obj    : (n_obj,) — per-object breakdown
    jump_ratio : float — max_step / median_step; >> 1 flags teleportation
    """
    n_obj = pos_rollout.shape[1]
    if pos_rollout.shape[0] < 3:
        nan = np.full(n_obj, np.nan)
        return np.nan, nan, np.nan

    vel = np.diff(pos_rollout, axis=0)                          # (N-1, n_obj, 2)
    acc = np.diff(vel, axis=0)                                  # (N-2, n_obj, 2)
    vel_mag = np.linalg.norm(vel, axis=-1)                      # (N-1, n_obj)
    acc_mag = np.linalg.norm(acc, axis=-1)                      # (N-2, n_obj)

    per_obj = acc_mag.mean(axis=0) / (vel_mag.mean(axis=0) + 1e-8)
    score = float(per_obj.mean())
    jump_ratio = float(
        (vel_mag.max(axis=0) / (np.median(vel_mag, axis=0) + 1e-8)).max()
    )
    return score, per_obj, jump_ratio

In [ ]:
assert ROLLOUT_N_CONTEXT + ROLLOUT_N_ROLLOUT <= T_frames, (
    f"N_CONTEXT + N_ROLLOUT ({ROLLOUT_N_CONTEXT + ROLLOUT_N_ROLLOUT}) > T_frames ({T_frames})"
)

with h5py.File(TEST_H5_PATH, "r") as _f:
    _n_test = _f["obs_intensity"].shape[0]
n_eval = min(ROLLOUT_N_EVAL, _n_test)

gt_scores, lin_scores, mlp_scores = [], [], []
gt_jumps,  lin_jumps,  mlp_jumps  = [], [], []

linear_extractor.eval()
mlp_extractor.eval()

for _idx in tqdm(range(n_eval), desc="rollout coherence"):
    _, _, _, _obs_np = nb_utils.load_sample(TEST_H5_PATH, _idx)
    with h5py.File(TEST_H5_PATH, "r") as _f:
        _pos = _f["positions"][_idx].astype(np.float32)  # (T, max_obj, 2)

    _h_ctx, _h_roll, _ = _collect_rollout(
        model, _obs_np, ROLLOUT_N_CONTEXT, ROLLOUT_N_ROLLOUT, DEVICE
    )
    _h_roll_t = torch.from_numpy(_h_roll).float().to(DEVICE)
    with torch.no_grad():
        _lin = linear_extractor(_h_roll_t).cpu().numpy()  # (N_roll, max_obj, 2)
        _mlp = mlp_extractor(_h_roll_t).cpu().numpy()

    _gt_roll = _pos[ROLLOUT_N_CONTEXT : ROLLOUT_N_CONTEXT + ROLLOUT_N_ROLLOUT]

    _s, _, _j = rollout_coherence(_gt_roll);  gt_scores.append(_s);  gt_jumps.append(_j)
    _s, _, _j = rollout_coherence(_lin);      lin_scores.append(_s); lin_jumps.append(_j)
    _s, _, _j = rollout_coherence(_mlp);      mlp_scores.append(_s); mlp_jumps.append(_j)

def _fmt(vals):
    a = np.array(vals)
    return f"{np.nanmean(a):.4f}  ±  {np.nanstd(a):.4f}"

print(f"Rollout coherence — mean relative acceleration  (lower = more linear)")
print(f"  context={ROLLOUT_N_CONTEXT}  rollout={ROLLOUT_N_ROLLOUT}  n_eval={n_eval}")
print(f"  GT     : {_fmt(gt_scores)}   jump ratio: {np.nanmean(gt_jumps):.2f}")
print(f"  Linear : {_fmt(lin_scores)}   jump ratio: {np.nanmean(lin_jumps):.2f}")
print(f"  MLP    : {_fmt(mlp_scores)}   jump ratio: {np.nanmean(mlp_jumps):.2f}")

In [ ]:
_gt_arr  = np.array(gt_scores)
_lin_arr = np.array(lin_scores)
_mlp_arr = np.array(mlp_scores)

_gt_j  = np.array(gt_jumps)
_lin_j = np.array(lin_jumps)
_mlp_j = np.array(mlp_jumps)

_labels = ["GT", "Linear", "MLP"]
_colors = ["#888888", LINEAR_COLOR, MLP_COLOR]
_x      = np.arange(len(_labels))
_w      = 0.5

fig, axes = plt.subplots(1, 2, figsize=(9, 4), facecolor=nb_viz._BG_HEX)
fig.suptitle(
    f"Rollout coherence  (context={ROLLOUT_N_CONTEXT}  rollout={ROLLOUT_N_ROLLOUT}  n={n_eval})",
    color=nb_viz._TEXT_COLOR, fontsize=12,
)

# ── MRA (mean relative acceleration) ─────────────────────────────────────────
ax = axes[0]
nb_viz.style_ax(ax)
_means = [np.nanmean(_gt_arr), np.nanmean(_lin_arr), np.nanmean(_mlp_arr)]
_stds  = [np.nanstd(_gt_arr),  np.nanstd(_lin_arr),  np.nanstd(_mlp_arr)]
bars = ax.bar(_x, _means, _w, yerr=_stds, color=_colors, alpha=0.85,
              error_kw=dict(ecolor=nb_viz._TICK_COLOR, capsize=4, linewidth=1.2))
ax.set_xticks(_x); ax.set_xticklabels(_labels, color=nb_viz._TEXT_COLOR, fontsize=10)
ax.set_ylabel("mean relative acceleration", color=nb_viz._TEXT_COLOR, fontsize=10)
ax.set_title("MRA  (lower = more linear)", color=nb_viz._TEXT_COLOR, fontsize=11)

# ── Jump ratio ────────────────────────────────────────────────────────────────
ax = axes[1]
nb_viz.style_ax(ax)
_jmeans = [np.nanmean(_gt_j), np.nanmean(_lin_j), np.nanmean(_mlp_j)]
_jstds  = [np.nanstd(_gt_j),  np.nanstd(_lin_j),  np.nanstd(_mlp_j)]
ax.bar(_x, _jmeans, _w, yerr=_jstds, color=_colors, alpha=0.85,
       error_kw=dict(ecolor=nb_viz._TICK_COLOR, capsize=4, linewidth=1.2))
ax.set_xticks(_x); ax.set_xticklabels(_labels, color=nb_viz._TEXT_COLOR, fontsize=10)
ax.set_ylabel("jump ratio  (max / median step)", color=nb_viz._TEXT_COLOR, fontsize=10)
ax.set_title("Jump ratio  (lower = smoother)", color=nb_viz._TEXT_COLOR, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
from pim.viz import _BG  # dark background array for waterfall pixels

# ── Load sample ───────────────────────────────────────────────────────────────
scene_v, obs_depth_v, obs_id_v, obs_intensity_v = nb_utils.load_sample(
    TEST_H5_PATH, ROLLOUT_VIZ_IDX
)
with h5py.File(TEST_H5_PATH, "r") as f:
    pos_v = f["positions"][ROLLOUT_VIZ_IDX].astype(np.float32)  # (T, max_obj, 2)

_n_ctx  = ROLLOUT_N_CONTEXT
_n_roll = ROLLOUT_N_ROLLOUT
cfg_v   = scene_v.config
T_v     = obs_intensity_v.shape[0]

h_ctx_v, h_roll_v, obs_roll_v = _collect_rollout(
    model, obs_intensity_v, _n_ctx, _n_roll, DEVICE
)

# ── Decode positions with both probes ─────────────────────────────────────────
linear_extractor.eval()
mlp_extractor.eval()
with torch.no_grad():
    _h_ctx_t  = torch.from_numpy(h_ctx_v ).float().to(DEVICE)
    _h_roll_t = torch.from_numpy(h_roll_v).float().to(DEVICE)
    lin_ctx_v  = linear_extractor(_h_ctx_t ).cpu().numpy()  # (n_ctx,  max_obj, 2)
    lin_roll_v = linear_extractor(_h_roll_t).cpu().numpy()  # (n_roll, max_obj, 2)
    mlp_ctx_v  = mlp_extractor(_h_ctx_t ).cpu().numpy()
    mlp_roll_v = mlp_extractor(_h_roll_t).cpu().numpy()

# ── Coherence scores for this sample ─────────────────────────────────────────
gt_roll_v = pos_v[_n_ctx : _n_ctx + _n_roll]
s_gt,  po_gt,  j_gt  = rollout_coherence(gt_roll_v)
s_lin, po_lin, j_lin = rollout_coherence(lin_roll_v)
s_mlp, po_mlp, j_mlp = rollout_coherence(mlp_roll_v)

# ── Build waterfall images ─────────────────────────────────────────────────────
wf_actual = make_waterfall(obs_depth_v, obs_id_v, obs_intensity_v, scene_v)  # (T, R, 4)

wf_pred = np.zeros_like(wf_actual)
wf_pred[:, :, :3] = _BG
wf_pred[:, :, 3]  = 1.0
wf_pred[:_n_ctx]  = wf_actual[:_n_ctx] * [1, 1, 1, 0.35]
wf_pred[:_n_ctx, :, 3] = 1.0
_gray = np.clip(obs_roll_v, 0.0, 1.0)
wf_pred[_n_ctx : _n_ctx + _n_roll, :, 0] = _gray
wf_pred[_n_ctx : _n_ctx + _n_roll, :, 1] = _gray
wf_pred[_n_ctx : _n_ctx + _n_roll, :, 2] = _gray

# ── Figure (3-panel, dark theme) ──────────────────────────────────────────────
_dbg = nb_viz._DARK_BG_HEX
_dtc = nb_viz._DARK_TEXT_COLOR
_dfe = nb_viz._DARK_FRUSTUM_EDGE

fig = plt.figure(figsize=(18, 5.5), facecolor=_dbg)
fig.subplots_adjust(left=0.04, right=0.98, top=0.86, bottom=0.10, wspace=0.14)
ax_wa = fig.add_subplot(1, 3, 1)
ax_wp = fig.add_subplot(1, 3, 2)
ax_2d = fig.add_subplot(1, 3, 3)
fig.suptitle(
    f"Sample {ROLLOUT_VIZ_IDX}  |  context={_n_ctx}  rollout={_n_roll}  |  "
    f"coherence (mra):  GT={s_gt:.3f}   linear={s_lin:.3f}   MLP={s_mlp:.3f}",
    color=_dtc, fontsize=11, y=0.97,
)

# ── Waterfall panels ──────────────────────────────────────────────────────────
_ext = [0, cfg_v.obs_res, T_v, 0]
for ax, wf, ttl in [
    (ax_wa, wf_actual, "actual"),
    (ax_wp, wf_pred,   f"predicted  (rollout from frame {_n_ctx})"),
]:
    nb_viz.style_ax(ax, dark=True)
    ax.imshow(wf, aspect="auto", origin="upper", interpolation="nearest", extent=_ext)
    ax.axhline(_n_ctx - 0.5, color="#fa8850", linewidth=1.0, linestyle="--", alpha=0.85)
    ax.set_title(ttl, color=_dtc, fontsize=11, pad=6)
    ax.set_xlabel("scan position", color=_dtc, fontsize=10)
    ax.set_ylabel("frame", color=_dtc, fontsize=10)
    ax.set_xlim(0, cfg_v.obs_res)
    ax.set_ylim(T_v, 0)

# ── 2D positions panel ────────────────────────────────────────────────────────
mx, my = 0.7, 0.7
ax_2d.set_xlim(-cfg_v.x_far - mx, cfg_v.x_far + mx)
ax_2d.set_ylim(cfg_v.y_near - my, cfg_v.y_far + my)
ax_2d.set_aspect("equal")
nb_viz.style_ax(ax_2d, dark=True)
ax_2d.set_xlabel("x", color=_dtc, fontsize=10)
ax_2d.set_ylabel("depth  y", color=_dtc, fontsize=10)
ax_2d.set_title(
    "2D positions  (faint = context  |  solid = rollout)\n×  linear      o  MLP",
    color=_dtc, fontsize=11, pad=6,
)

corners = np.array([
    [-cfg_v.x_near, cfg_v.y_near], [cfg_v.x_near, cfg_v.y_near],
    [cfg_v.x_far,   cfg_v.y_far],  [-cfg_v.x_far, cfg_v.y_far],
])
ax_2d.add_patch(MPoly(corners, closed=True, fill=False,
                       edgecolor=_dfe, linewidth=1.5, zorder=1))

for obj in range(scene_v.positions.shape[1]):
    color = scene_v.colors[obj]

    # GT: full trajectory faint, rollout window highlighted
    ax_2d.plot(pos_v[:, obj, 0], pos_v[:, obj, 1],
               color=color, linewidth=1.0, alpha=0.30, zorder=2)
    ax_2d.plot(pos_v[_n_ctx : _n_ctx + _n_roll, obj, 0],
               pos_v[_n_ctx : _n_ctx + _n_roll, obj, 1],
               color=color, linewidth=1.8, alpha=0.90, zorder=3)

    # Linear probe: context (faint ×), rollout (solid ×)
    ax_2d.scatter(lin_ctx_v[:, obj, 0],  lin_ctx_v[:, obj, 1],
                  color=color, s=10, marker="x", alpha=0.25, zorder=3)
    ax_2d.scatter(lin_roll_v[:, obj, 0], lin_roll_v[:, obj, 1],
                  color=color, s=30, marker="x", alpha=0.95, zorder=4)

    # MLP probe: context (faint o), rollout (solid o)
    ax_2d.scatter(mlp_ctx_v[:, obj, 0],  mlp_ctx_v[:, obj, 1],
                  color=color, s=10, marker="o", alpha=0.25, zorder=3)
    ax_2d.scatter(mlp_roll_v[:, obj, 0], mlp_roll_v[:, obj, 1],
                  color=color, s=30, marker="o", alpha=0.95, zorder=4)

plt.tight_layout()
plt.show()

print(f"\nSample {ROLLOUT_VIZ_IDX}  |  mra = mean relative acceleration  |  jump = max/median step")
for name, score, per_obj, jump in [
    ("GT    ", s_gt,  po_gt,  j_gt),
    ("Linear", s_lin, po_lin, j_lin),
    ("MLP   ", s_mlp, po_mlp, j_mlp),
]:
    obj_str = "  ".join(f"obj{i}:{v:.3f}" for i, v in enumerate(per_obj))
    print(f"  {name}  mra={score:.4f}  jump={jump:.2f}  [{obj_str}]")

---
## 5 — Counterfactual Controllability

Evaluates whether the GRU can be **steered** mid-sequence by directly editing its hidden state.

**Protocol:**
1. Teacher-force the GRU on pre-edit observations `obs[0 : edit_frame]` → hidden state `h`
2. Decompose `h = h_parallel + h_perp` via the linear probe's row-space projection
3. Replace the parallel component: `h' = A⁺(y' − b) + h_perp`, where `y'` is the post-edit position
4. Decode `h'` → first predicted observation; autoregressive rollout from there (model never sees ground truth)
5. Compare steered vs. unsteered MSE against post-edit observations

**Math.** Let `A = linear_extractor.linear.weight` (`[4, 256]`), `b` = bias.
- Pseudoinverse: `A⁺ = Aᵀ(AAᵀ)⁻¹` (shape `[256, 4]`; inverts a `[4, 4]` matrix — stable because hidden_size ≫ 4)
- Decomposition: `h_parallel = A⁺(Ah)`, `h_perp = h − h_parallel`
- Edit: `h' = A⁺(y' − b) + h_perp`  →  `Ah' + b = AA⁺(y'−b) + 0 + b = y'` ✓

In [ ]:
EDITS_H5_PATH  = "../datasets/3_fixed_refl_inview_brighter_edits/dataset.h5"   # path to an edits dataset

CTRL_N_ROLLOUT = 15    # AR steps after edit_frame to evaluate
CTRL_N_EVAL    = 200   # samples for population-level MSE
CTRL_VIZ_IDX   = 2

In [ ]:
# ── Probe decomposition helpers ───────────────────────────────────────────────

def _probe_matrix():
    """Extract (A, b, A_pinv) from the trained linear extractor.

    A      : [out, H]  = [4, 256]  — probe weight matrix
    b      : [out]     = [4]       — probe bias
    A_pinv : [H, out]  = [256, 4]  — Moore-Penrose pseudoinverse  A⁺ = Aᵀ(AAᵀ)⁻¹
    """
    A = linear_extractor.linear.weight.detach().cpu()   # [4, 256]
    b = linear_extractor.linear.bias.detach().cpu()     # [4]
    AAt = A @ A.t()                                     # [4, 4]
    A_pinv = A.t() @ torch.linalg.inv(AAt)             # [256, 4]
    return A, b, A_pinv


def _decompose_h(h, A, A_pinv):
    """Decompose h into row-space (parallel) and null-space (perp) of A.
    h : [H] — single hidden state vector.
    """
    h_par = A_pinv @ (A @ h)   # projection onto row(A)
    return h_par, h - h_par


def _inject_position(h, A, A_pinv, b, y_flat):
    """Return h' such that the probe reads y_flat: A h' + b = y_flat.
    y_flat : [out] — desired flattened position vector.
    """
    _, h_perp = _decompose_h(h, A, A_pinv)
    return A_pinv @ (y_flat - b) + h_perp


# ── Sanity checks ─────────────────────────────────────────────────────────────
_A, _b, _Apinv = _probe_matrix()
_I4 = _A @ _Apinv
print(f"A A⁺ ≈ I₄  (max |err|) : {(_I4 - torch.eye(4)).abs().max().item():.2e}")
print(f"Condition number of AAᵀ : {torch.linalg.cond(_A @ _A.t()).item():.2f}")

In [ ]:
@torch.no_grad()
def _edits_rollout(model, obs_np, edit_frame, y_flat_target, A, A_pinv, b, n_rollout, device):
    """Teacher-force obs[0:edit_frame] → h_context, then do two AR rollouts.

    Unsteered : AR from original h_context (no intervention).
    Steered   : edit h → h', decode h' → first prediction, AR from there.
                Model never receives post-edit ground-truth observations.

    Parameters
    ----------
    obs_np        : (T, R) numpy array — full observation sequence
    edit_frame    : int — context length; rollout begins at this frame
    y_flat_target : [out] cpu tensor — desired probe output (post-edit positions, flattened)
    n_rollout     : number of AR steps after edit_frame

    Returns
    -------
    unsteered : (n_rollout, R) numpy
    steered   : (n_rollout, R) numpy
    h_steered : (n_rollout + 1, H) numpy
                index 0  = h' itself (probe(h') == y_flat_target exactly)
                index 1+ = hidden states after each subsequent AR step
    """
    obs = torch.from_numpy(obs_np).float().to(device)   # (T, R)
    h = None
    for t in range(edit_frame):
        pred, h = model.step(obs[t].unsqueeze(0), h)
    # pred == decoder(h[-1, 0]) — model's prediction from last pre-edit observation

    # ── unsteered: continue AR from original h ────────────────────────────────
    unsteered, h_u, x = [], h, pred
    for _ in range(n_rollout):
        pred_u, h_u = model.step(x, h_u)
        unsteered.append(pred_u[0].cpu())
        x = pred_u

    # ── steered: inject edited hidden state, re-decode, then AR ──────────────
    h_vec    = h[-1, 0].cpu()                                # [H]
    h_edited = _inject_position(h_vec, A, A_pinv, b, y_flat_target)
    h_s      = h.clone()
    h_s[-1, 0] = h_edited.to(device)                        # edit last GRU layer only
    pred_s_first = model.decoder(h_s[-1])                    # (1, R) — decode h'

    # h' itself is index 0 so probe(h_steered[0]) == y_flat_target exactly
    h_steered_list = [h_edited]
    steered, x = [], pred_s_first
    for _ in range(n_rollout):
        pred_s, h_s = model.step(x, h_s)
        steered.append(pred_s[0].cpu())
        h_steered_list.append(h_s[-1, 0].cpu())             # [H]
        x = pred_s

    return (
        torch.stack(unsteered).numpy(),       # (n_rollout, R)
        torch.stack(steered).numpy(),         # (n_rollout, R)
        torch.stack(h_steered_list).numpy(),  # (n_rollout + 1, H)
    )

In [ ]:
# ── Load edits sample ─────────────────────────────────────────────────────────
_scene_ev, _obs_depth_ev, _obs_id_ev, _obs_intensity_ev = nb_utils.load_sample(
    EDITS_H5_PATH, CTRL_VIZ_IDX
)
with h5py.File(EDITS_H5_PATH, "r") as f:
    _edit_frame_v = int(f["edit_frame"][CTRL_VIZ_IDX])
    _edit_obj_v   = int(f["edit_object"][CTRL_VIZ_IDX])
    _edit_val_v   = f["edit_value"][CTRL_VIZ_IDX].astype(np.float32)    # (2,)
    _pos_ev       = f["positions"][CTRL_VIZ_IDX].astype(np.float32)     # (T, max_obj, 2)

_y_flat_v = torch.from_numpy(_pos_ev[_edit_frame_v].reshape(-1))        # [4]
_ef = _edit_frame_v
_nr = CTRL_N_ROLLOUT

print(f"Sample {CTRL_VIZ_IDX}  |  edit_frame={_ef}  |  edit_object={_edit_obj_v}")
print(f"Edit position  : ({_edit_val_v[0]:.3f}, {_edit_val_v[1]:.3f})")

# ── Run rollouts ──────────────────────────────────────────────────────────────
_A, _b, _Apinv = _probe_matrix()

_unsteered_v, _steered_v, _h_strd_v = _edits_rollout(
    model, _obs_intensity_ev, _ef, _y_flat_v,
    _A, _Apinv, _b, _nr, DEVICE,
)
# _h_strd_v shape: (n_rollout + 1, H)
#   index 0  = h' (injected state; probe reads y_flat_v exactly)
#   index 1+ = post-step hidden states

# ── Build waterfall images ─────────────────────────────────────────────────────
T_ev = _obs_intensity_ev.shape[0]
R_ev = _obs_intensity_ev.shape[1]
_wf_actual = make_waterfall(_obs_depth_ev, _obs_id_ev, _obs_intensity_ev, _scene_ev)


def _pred_wf(rollout_obs):
    wf = np.zeros((T_ev, R_ev, 4), dtype=np.float32)
    wf[:, :, :3] = _BG
    wf[:, :, 3]  = 1.0
    wf[:_ef]          = _wf_actual[:_ef] * [1, 1, 1, 0.35]
    wf[:_ef, :, 3]    = 1.0
    gray = np.clip(rollout_obs[:_nr], 0.0, 1.0)
    wf[_ef : _ef + _nr, :, 0] = gray
    wf[_ef : _ef + _nr, :, 1] = gray
    wf[_ef : _ef + _nr, :, 2] = gray
    return wf


_dbg = nb_viz._DARK_BG_HEX
_dtc = nb_viz._DARK_TEXT_COLOR
_dfe = nb_viz._DARK_FRUSTUM_EDGE
_ext_ev = [0, R_ev, T_ev, 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), facecolor=_dbg)
fig.suptitle(
    f"Counterfactual Controllability  |  sample {CTRL_VIZ_IDX}  "
    f"|  edit frame={_ef}  obj={_edit_obj_v}  rollout={_nr} steps",
    color=_dtc, fontsize=11, y=1.01,
)
for ax, wf, ttl in zip(
    axes,
    [_wf_actual, _pred_wf(_unsteered_v), _pred_wf(_steered_v)],
    ["actual  (post-edit GT)", "unsteered  (no h edit)", "steered  (h' injected)"],
):
    nb_viz.style_ax(ax, dark=True)
    ax.imshow(wf, aspect="auto", origin="upper", interpolation="nearest", extent=_ext_ev)
    ax.axhline(_ef - 0.5, color="#fa8850", linewidth=1.0, linestyle="--", alpha=0.85)
    ax.set_title(ttl, color=_dtc, fontsize=11, pad=6)
    ax.set_xlabel("scan position", color=_dtc, fontsize=10)
    ax.set_ylabel("frame", color=_dtc, fontsize=10)
    ax.set_xlim(0, R_ev)
    ax.set_ylim(T_ev, 0)
plt.tight_layout()
plt.show()

# ── Decode 2D positions from steered hidden states ───────────────────────────
# _h_strd_v has n_rollout+1 rows: h' at index 0, then post-step states.
# So _lin_strd[0] == y_flat_v (exactly) and _lin_strd[s] ↔ frame _ef+s.
linear_extractor.eval()
mlp_extractor.eval()
with torch.no_grad():
    _h_strd_t = torch.from_numpy(_h_strd_v).float().to(DEVICE)    # (n_roll+1, H)
    _lin_strd = linear_extractor(_h_strd_t).cpu().numpy()          # (n_roll+1, max_obj, 2)
    _mlp_strd = mlp_extractor(_h_strd_t).cpu().numpy()

# ── Per-step decoded positions vs GT ─────────────────────────────────────────
# Step 0 = edit_frame (h' injected), probe should match GT exactly.
# Steps 1..n_rollout = subsequent AR steps.
_n_obj = _scene_ev.positions.shape[1]
_hdr = f"{'step':>4}  {'frame':>5}  " + "  ".join(
    f"{'gt_obj'+str(o)+' (x,y)':>16}  {'lin_obj'+str(o)+' (x,y)':>16}"
    for o in range(_n_obj)
)
print(f"\nDecoded positions (linear probe) vs GT  —  steered rollout:")
print(_hdr)
for _s in range(_nr + 1):
    _row = f"{_s:>4}  {_ef + _s:>5}  "
    for _o in range(_n_obj):
        _g = _pos_ev[_ef + _s, _o]
        _l = _lin_strd[_s, _o]
        _row += f"({_g[0]:5.2f},{_g[1]:5.2f})  ({_l[0]:5.2f},{_l[1]:5.2f})  "
    print(_row)

# ── 2D positions plot ─────────────────────────────────────────────────────────
_cfg_ev = _scene_ev.config
_mx, _my = 0.7, 0.7

fig, ax = plt.subplots(figsize=(5.5, 7), facecolor=_dbg)
nb_viz.style_ax(ax, dark=True)
ax.set_xlim(-_cfg_ev.x_far - _mx, _cfg_ev.x_far + _mx)
ax.set_ylim(_cfg_ev.y_near - _my, _cfg_ev.y_far + _my)
ax.set_aspect("equal")
ax.set_xlabel("x", color=_dtc, fontsize=10)
ax.set_ylabel("depth  y", color=_dtc, fontsize=10)
ax.set_title(
    f"2D positions  |  sample {CTRL_VIZ_IDX}\n"
    f"solid=GT   ×=linear   o=MLP  (rollout only)",
    color=_dtc, fontsize=11, pad=6,
)

_corners = np.array([
    [-_cfg_ev.x_near, _cfg_ev.y_near], [_cfg_ev.x_near, _cfg_ev.y_near],
    [_cfg_ev.x_far,   _cfg_ev.y_far],  [-_cfg_ev.x_far, _cfg_ev.y_far],
])
ax.add_patch(MPoly(_corners, closed=True, fill=False,
                   edgecolor=_dfe, linewidth=1.5, zorder=1))

for _obj in range(_n_obj):
    _c = _scene_ev.colors[_obj]

    # GT: full context faint, post-edit rollout bright (_nr+1 frames including edit_frame)
    ax.plot(_pos_ev[:_ef, _obj, 0], _pos_ev[:_ef, _obj, 1],
            color=_c, linewidth=1.0, alpha=0.25, zorder=2)
    ax.plot(_pos_ev[_ef : _ef + _nr + 1, _obj, 0], _pos_ev[_ef : _ef + _nr + 1, _obj, 1],
            color=_c, linewidth=1.8, alpha=0.9, zorder=3)

    # Mark teleport: pre-edit position and post-edit position
    ax.scatter(_pos_ev[_ef - 1, _obj, 0], _pos_ev[_ef - 1, _obj, 1],
               color=_c, s=50, marker="o", zorder=5, alpha=0.6)
    ax.scatter(_pos_ev[_ef, _obj, 0], _pos_ev[_ef, _obj, 1],
               color="#fa8850", s=80, marker="*", zorder=6)

    # Probe-decoded steered positions (n_rollout+1 points, frames _ef.._ef+_nr)
    ax.scatter(_lin_strd[:, _obj, 0], _lin_strd[:, _obj, 1],
               color=_c, s=30, marker="x", alpha=0.9, zorder=4)
    ax.scatter(_mlp_strd[:, _obj, 0], _mlp_strd[:, _obj, 1],
               color=_c, s=30, marker="o", alpha=0.9, zorder=4)

ax.scatter([], [], color="white", marker="x", s=30, label="linear probe")
ax.scatter([], [], color="white", marker="o", s=30, label="MLP probe")
ax.scatter([], [], color="#fa8850", marker="*", s=60, label=f"edit point  (frame {_ef})")
ax.legend(labelcolor=_dtc, facecolor=_dbg, edgecolor=nb_viz._DARK_TICK_COLOR, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── X / Y position trajectories: GT vs probe predictions (steered rollout) ───
# Shows a few pre-edit context frames (GT only) + full rollout window
# so the discontinuity at the edit is visible.
# _lin_strd / _mlp_strd have n_rollout+1 rows:
#   index 0 = h' (probe == y_flat exactly, frame _ef)
#   index 1+ = post-step states (frames _ef+1 .. _ef+_nr)

_n_ctx_show = min(8, _ef)
_ctx_frames  = np.arange(_ef - _n_ctx_show, _ef)          # absolute frame indices
_roll_frames = np.arange(_ef, _ef + _nr + 1)              # n_rollout+1 frames

fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor=nb_viz._BG_HEX)
fig.suptitle(
    f"Sample {CTRL_VIZ_IDX} — steered rollout positions  "
    f"(edit at frame {_ef}, obj {_edit_obj_v})\n"
    "solid=GT   ×=linear   o=MLP  |  faint region = pre-edit context  |  "
    "step 0 = injected h' (linear × should be exact)",
    color=nb_viz._TEXT_COLOR, fontsize=11,
)

for ax, coord, label in zip(axes, [0, 1], ["x", "y (depth)"]):
    nb_viz.style_ax(ax)

    # Vertical line at edit frame
    ax.axvline(_ef - 0.5, color=nb_viz._TICK_COLOR, linewidth=1.0,
               linestyle="--", alpha=0.5)

    for _obj in range(_n_obj):
        color = nb_viz.plot_color(_scene_ev.colors[_obj])

        # Pre-edit context: GT only (faint)
        ax.plot(_ctx_frames, _pos_ev[_ef - _n_ctx_show : _ef, _obj, coord],
                color=color, linewidth=1.5, alpha=0.3)

        # Rollout: GT solid line (n_rollout+1 frames, _ef .. _ef+_nr)
        ax.plot(_roll_frames, _pos_ev[_ef : _ef + _nr + 1, _obj, coord],
                color=color, linewidth=1.8, alpha=0.9,
                label=f"obj {_obj} GT")

        # Rollout: linear probe decoded (×) — index 0 should sit exactly on GT
        ax.scatter(_roll_frames, _lin_strd[:, _obj, coord],
                   color=color, s=20, marker="x", alpha=0.9,
                   label=f"obj {_obj} linear")

        # Rollout: MLP probe decoded (○)
        ax.scatter(_roll_frames, _mlp_strd[:, _obj, coord],
                   color=color, s=20, marker="o", alpha=0.9,
                   label=f"obj {_obj} MLP")

    ax.set_xlabel("frame", color=nb_viz._TEXT_COLOR, fontsize=10)
    ax.set_ylabel(label, color=nb_viz._TEXT_COLOR, fontsize=10)
    ax.set_title(label, color=nb_viz._TEXT_COLOR, fontsize=11)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend(
        labelcolor=nb_viz._TEXT_COLOR,
        facecolor=nb_viz._BG_HEX,
        edgecolor=nb_viz._TICK_COLOR,
        fontsize=7, ncol=2,
    )

plt.tight_layout()
plt.show()

In [ ]:
# ── Population MSE: steered vs unsteered ─────────────────────────────────────
_A, _b, _Apinv = _probe_matrix()

_mse_unst = np.zeros(CTRL_N_ROLLOUT)
_mse_strd = np.zeros(CTRL_N_ROLLOUT)
_count = 0

with h5py.File(EDITS_H5_PATH, "r") as f:
    _edit_frames = f["edit_frame"][:CTRL_N_EVAL]    # (N,) int32
    _positions   = f["positions"][:CTRL_N_EVAL]      # (N, T, max_obj, 2)

for _idx in tqdm(range(CTRL_N_EVAL), desc="controllability eval"):
    _, _, _, _obs_np = nb_utils.load_sample(EDITS_H5_PATH, _idx)
    _ef_i   = int(_edit_frames[_idx])
    _y_flat = torch.from_numpy(_positions[_idx, _ef_i].reshape(-1).astype(np.float32))

    _n_roll = min(CTRL_N_ROLLOUT, _obs_np.shape[0] - _ef_i)
    if _n_roll <= 0:
        continue

    _unst, _strd, _ = _edits_rollout(
        model, _obs_np, _ef_i, _y_flat, _A, _Apinv, _b, _n_roll, DEVICE
    )
    _gt = _obs_np[_ef_i : _ef_i + _n_roll]     # (n_roll, R)

    _mse_unst[:_n_roll] += ((_unst - _gt) ** 2).mean(axis=1)
    _mse_strd[:_n_roll] += ((_strd - _gt) ** 2).mean(axis=1)
    _count += 1

_mse_unst /= _count
_mse_strd /= _count

print(f"Evaluated {_count} samples")
print(f"Mean MSE (unsteered) over rollout : {_mse_unst.mean():.6f}")
print(f"Mean MSE (steered)   over rollout : {_mse_strd.mean():.6f}")

In [ ]:
steps = np.arange(1, CTRL_N_ROLLOUT + 1)

fig, ax = plt.subplots(figsize=(8, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.plot(steps, _mse_unst, color="#D55E00", linewidth=1.8, label="unsteered")
ax.plot(steps, _mse_strd, color="#0072B2", linewidth=1.8, label="steered  (h' injected)")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_xlabel("step after edit", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel("mean MSE", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title(
    f"Controllability: steered vs unsteered  ({_count} samples)",
    color=nb_viz._TEXT_COLOR, fontsize=12,
)
ax.legend(
    labelcolor=nb_viz._TEXT_COLOR,
    facecolor=nb_viz._BG_HEX,
    edgecolor=nb_viz._TICK_COLOR,
    fontsize=10,
)
plt.tight_layout()
plt.show()